In [9]:
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
from sentence_transformers import SentenceTransformer
from pathlib import Path

In [2]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embed_dim = embedder.get_embedding_dimension()
    

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
input_filename = Path("../data/raw/arxiv-metadata-oai-snapshot.parquet")
assert input_filename.exists(), f"Input file {input_filename} does not exist."

batch_size = 100

In [7]:
data = pq.ParquetFile(input_filename)
total_rows = data.metadata.num_rows

In [8]:
print(total_rows)

3028029


In [17]:
def embed_batch(s: pl.Series) -> pl.Series:
    texts = s.to_list()
    embeddings = embedder.encode(texts, batch_size=batch_size, precision="float32", show_progress_bar=False).tolist()
    # pbar.update(len(texts))
    return pl.Series(embeddings, dtype=pl.Array(pl.Float32, embedder.get_embedding_dimension()))

In [19]:
for batch in data.iter_batches(batch_size, columns=["id", "title", "abstract"]):
    
    batch_df = pl.from_arrow(batch)
    
    batch_df = (
        pl.from_arrow(batch)
        .filter(pl.col("abstract").str.len_chars() > 150)
        .select(
            pl.col("id"),
            (pl.lit("Title: ") + pl.col("title") + pl.lit(" | Abstract: ") + pl.col("abstract")).alias("content")
        )
    )
    
    batch_len = batch_df.height
    
    if batch_len == 0:
        continue
    
    batch_df = batch_df.with_columns(
        pl.col("content").map_batches(embed_batch, return_dtype=pl.Array(pl.Float32, embed_dim)).alias("embedding")
    )
    
    break

In [21]:
batch_df

id,content,embedding
str,str,"array[f32, 384]"
"""0704.0001""","""Title: Calculation of prompt d…","[-0.158678, -0.010544, … 0.049717]"
"""0704.0002""","""Title: Sparsity-certifying Gra…","[0.011342, 0.034085, … 0.021261]"
"""0704.0003""","""Title: The evolution of the Ea…","[0.031139, -0.031973, … 0.004312]"
"""0704.0004""","""Title: A determinant of Stirli…","[-0.068851, -0.018202, … 0.000149]"
"""0704.0005""","""Title: From dyadic $\Lambda_{\…","[0.035553, 0.003334, … -0.026612]"
…,…,…
"""0704.0096""","""Title: Much ado about 248 | Ab…","[-0.083472, -0.025992, … 0.026993]"
"""0704.0097""","""Title: Conformal Field Theory …","[-0.108841, -0.001429, … 0.051152]"
"""0704.0098""","""Title: Sparsely-spread CDMA - …","[0.005602, -0.021998, … 0.048442]"
